# Week 2 Day 2: Gradio UIs for Local LLMs
Today we will learn how to build interactive web interfaces for our local models using **Gradio**.

## What You'll Learn:
1. **Gradio Basics:** How to build a simple web interface for a basic Python function.
2. **Local LLM Chatbot:** Building a beautiful chat interface that connects directly to our local model.

In [1]:
import sys
import subprocess

# 1. Automatically install gradio if it is missing
try:
    import gradio as gr
    print("✅ Gradio is already installed.")
except ImportError:
    print("📦 Gradio not found. Installing now...")
    # Installs Gradio into the active environment
    subprocess.check_call([sys.executable, "-m", "pip", "install", "gradio"])
    import gradio as gr

# 2. Import OpenAI to connect to our local Ollama
from openai import OpenAI

print("All imports successful! Ready to build UIs.")

✅ Gradio is already installed.
All imports successful! Ready to build UIs.


## 1. Gradio Basics
Let's build a simple UI for a regular Python function that takes a word and converts it to uppercase ("shouting").

We will use:
- `inputs="textbox"` (provides a text box for the user to type in).
- `outputs="textbox"` (provides a text box to display the result).
- `.launch()` (runs the web server).

In [2]:
# 1. Define a standard Python function
def shout(text):
    print(f"Python function received input: '{text}'")
    return text.upper()

# 2. Build the web interface
demo = gr.Interface(
    fn=shout,
    inputs="textbox",
    outputs="textbox",
    flagging_mode="never" # Disables the default "Flag" feedback button for simplicity
)

# 3. Launch the interface (this will display the UI directly inside your notebook!)
demo.launch()

* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


## 2. Building a Local AI Chatbot UI

Gradio's `gr.ChatInterface()` expects a chat function with two parameters:

1. **`message`:** The new message string typed by the user.
2. **`history`:** A list containing the entire conversation history.

> **Note:** Starting with **Gradio 5**, the `history` is already provided in the OpenAI chat message format as a list of dictionaries:
>
> ```python
> [
>     {"role": "user", "content": "Hello"},
>     {"role": "assistant", "content": "Hi! How can I help you?"},
>     {"role": "user", "content": "Tell me about LLMs."}
> ]
> ```
>
> In previous versions of Gradio, `history` was stored as a list of pairs:
>
> ```python
> [
>     [user_msg1, assistant_msg1],
>     [user_msg2, assistant_msg2],
>     ...
> ]
> ```
>
> which required looping through the history and manually converting each message into the OpenAI format.



In [3]:
# 1. Initialize the local OpenAI client (talking to local Ollama)
client = OpenAI(base_url="http://localhost:11434/v1", api_key="ollama")


# 2. Define the chat wrapper function
def local_chatbot(message, history):
    # Set up our System Instruction
    messages = [
        {"role": "system", "content": "You are a friendly, helpful local AI assistant"}
    ]

    # Gradio 5 history is already a list of {"role": "...", "content": "..."} dicts!
    # We can just append the whole history list directly.
    messages.extend(history)

    # Append the current user message at the end
    messages.append({"role": "user", "content": message})

    # Call the local model
    response=client.chat.completions.create(
        model="llama3.2:1b",
        messages=messages
    )

    # Return the reply string
    return  response.choices[0].message.content

# 3. Launch the chat interface!
chatbot_ui = gr.ChatInterface(
    fn=local_chatbot,
    title="Local Llama Chatbot",
    description="Ask anything! Powered by Ollama and GHradio."
)

chatbot_ui.launch()

* Running on local URL:  http://127.0.0.1:7861
* To create a public link, set `share=True` in `launch()`.
